# 01 — 解压、读取和初步识别原始数据

本 Notebook 完成以下任务：
1. 自动解压 7 个 CSMAR `.zip` 文件到 `data/raw/`
2. 读取每个数据文件，查看结构
3. 建立初步变量字典 `data/dict/variable_dictionary.csv`

In [8]:
from pathlib import Path
import zipfile
import pandas as pd
import shutil

BASE = Path.cwd()
ZIP_DIR = BASE / 'data' / 'data_raw_zip'
RAW_DIR = BASE / 'data' / 'raw'
DICT_DIR = BASE / 'data' / 'dict'
RAW_DIR.mkdir(parents=True, exist_ok=True)
DICT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── 自动创建项目目录结构 ──
DIRS = [
    BASE / "data" / "raw",
    BASE / "data" / "dict",
    BASE / "data" / "clean",
    BASE / "data" / "combined",
    BASE / "data" / "temp",
    BASE / "output" / "tables",
    BASE / "output" / "figures",
]
for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)
print(f"项目目录结构已自动创建：{len(DIRS)} 个目录")


In [9]:
zip_files = sorted(ZIP_DIR.glob('*.zip'))
print(f'找到 {len(zip_files)} 个原始 zip 文件：')
for zf in zip_files:
    print(f'  {zf.name}')

找到 7 个原始 zip 文件：
  CSMAR常用变量-2000-2024.zip
  上市公司基本信息变更表2000-2024.zip
  上市公司基本信息年度表.zip
  利润表-现金流量表-2000-2010.zip
  利润表-现金流量表-2011-2024.zip
  资产负债表-2000-2010.zip
  资产负债表-2011-2024.zip


## 1. 自动解压

遍历每个 `.zip`，解压其中的 `.xlsx` 到 `data/raw/`。
注意：部分文件（如利润表、资产负债表）按年份分段，解压时用年份后缀区分同名文件。

In [10]:
for zf in zip_files:
    # Use zip file's stem as the output xlsx name (avoids name conflicts)
    stem = zf.stem
    with zipfile.ZipFile(zf, 'r') as z:
        for info in z.infolist():
            if not info.filename.endswith('.xlsx') or info.filename.startswith('~'):
                continue
            # Rename to zip-stem.xlsx
            dst = RAW_DIR / f'{stem}.xlsx'
            z.extract(info, RAW_DIR)
            src = RAW_DIR / info.filename
            if src != dst:
                if dst.exists():
                    dst.unlink()
                src.rename(dst)
            print(f'  解压: {zf.name}  →  {dst.name}')

# Clean up subdirectories
for item in list(RAW_DIR.iterdir()):
    if item.is_dir():
        shutil.rmtree(item)

xlsx_files = sorted(RAW_DIR.glob('*.xlsx'))
print(f'\n解压后 ({len(xlsx_files)} 个文件):')
for f in xlsx_files:
    print(f'  [{f.stat().st_size/1024:>8.0f} KB] {f.name}')

  解压: CSMAR常用变量-2000-2024.zip  →  CSMAR常用变量-2000-2024.xlsx
  解压: 上市公司基本信息变更表2000-2024.zip  →  上市公司基本信息变更表2000-2024.xlsx
  解压: 上市公司基本信息年度表.zip  →  上市公司基本信息年度表.xlsx
  解压: 利润表-现金流量表-2000-2010.zip  →  利润表-现金流量表-2000-2010.xlsx
  解压: 利润表-现金流量表-2011-2024.zip  →  利润表-现金流量表-2011-2024.xlsx
  解压: 资产负债表-2000-2010.zip  →  资产负债表-2000-2010.xlsx
  解压: 资产负债表-2011-2024.zip  →  资产负债表-2011-2024.xlsx

解压后 (7 个文件):
  [   10841 KB] CSMAR常用变量-2000-2024.xlsx
  [    9081 KB] 上市公司基本信息变更表2000-2024.xlsx
  [   26853 KB] 上市公司基本信息年度表.xlsx
  [   10438 KB] 利润表-现金流量表-2000-2010.xlsx
  [   20459 KB] 利润表-现金流量表-2011-2024.xlsx
  [    9152 KB] 资产负债表-2000-2010.xlsx
  [   17716 KB] 资产负债表-2011-2024.xlsx


## 2. 读取并识别数据结构

每个 xlsx 的前三行：Row 0 = 英文列名，Row 1 = 中文标签，Row 2 = 单位。
实际数据从 Row 3 开始。读取时 `skiprows=[1, 2]` 跳过中文标签和单位行，保留英文列名。

In [11]:
for f in xlsx_files:
    print(f'\n{"="*60}')
    print(f'  文件: {f.name}')
    print(f'{"="*60}')
    
    # Show column names (skip first 2 rows for actual data)
    df_meta = pd.read_excel(f, nrows=2, header=None)
    print(f'  中文标签行: {list(df_meta.iloc[0])}')
    
    # Skip Chinese label and unit rows, keep English codes as headers
    df = pd.read_excel(f, skiprows=[1,2], nrows=3)
    cols_df = pd.read_excel(f, skiprows=[1,2], nrows=0)
    print(f'  列数: {len(cols_df.columns)}')
    for i, col in enumerate(cols_df.columns):
        print(f'    [{i:2d}] {col}')
    print(f'\n  前 3 行数据:')
    print(df.to_string())


  文件: CSMAR常用变量-2000-2024.xlsx


d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  中文标签行: ['Stkcd', 'accper', 'stknme', 'AnaAttention', 'Audittyp', 'InternationalBig4', 'Ysmvosd', 'Ysmvttl', 'Yretwd', 'PropertyRightsNature', 'Seperation', 'ActualControllerNatureID', 'OwnershipProportion', 'ControlProportion', 'Shrcr1', 'Shrhfd5', 'Shrz', 'FundHoldProportion', 'QFIIHoldProportion', 'BrokerHoldProportion', 'BankHoldProportion', 'NonFinanceHoldProportion', 'InsInvestorProp', 'StaffNumber', 'ConcurrentPosition', 'Boardsize2', 'ExecutivesNumber', 'IndDirector', 'SumSalary', 'TOP3SumSalary', 'Ynshrtrd', 'DirectorHoldshares', 'ManageHoldshares']


d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  列数: 33
    [ 0] Stkcd
    [ 1] accper
    [ 2] stknme
    [ 3] AnaAttention
    [ 4] Audittyp
    [ 5] InternationalBig4
    [ 6] Ysmvosd
    [ 7] Ysmvttl
    [ 8] Yretwd
    [ 9] PropertyRightsNature
    [10] Seperation
    [11] ActualControllerNatureID
    [12] OwnershipProportion
    [13] ControlProportion
    [14] Shrcr1
    [15] Shrhfd5
    [16] Shrz
    [17] FundHoldProportion
    [18] QFIIHoldProportion
    [19] BrokerHoldProportion
    [20] BankHoldProportion
    [21] NonFinanceHoldProportion
    [22] InsInvestorProp
    [23] StaffNumber
    [24] ConcurrentPosition
    [25] Boardsize2
    [26] ExecutivesNumber
    [27] IndDirector
    [28] SumSalary
    [29] TOP3SumSalary
    [30] Ynshrtrd
    [31] DirectorHoldshares
    [32] ManageHoldshares

  前 3 行数据:
   Stkcd  accper stknme  AnaAttention Audittyp  InternationalBig4      Ysmvosd      Ysmvttl    Yretwd  PropertyRightsNature  Seperation  ActualControllerNatureID  OwnershipProportion  ControlProportion  Shrcr1  Shrhfd5  Shrz 

...

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  列数: 36
    [ 0] code
    [ 1] stknme
    [ 2] listingDate
    [ 3] EndDate
    [ 4] FS_Comins-B001101000
    [ 5] FS_Comins-Bbd1102203
    [ 6] FS_Comins-B001201000
    [ 7] FS_Comins-B001207000
    [ 8] FS_Comins-B001209000
    [ 9] FS_Comins-B001210000
    [10] FS_Comins-B001211000
    [11] FS_Comins-B001300000
    [12] FS_Comins-B001000000
    [13] FS_Comins-B002000000
    [14] FS_Comins-B003000000
    [15] FS_Comins-B001216000
    [16] FS_Comscfd-C001001000
    [17] FS_Comscfd-C0d1008000
    [18] FS_Comscfd-C0f1009000
    [19] FS_Comscfd-C001012000
    [20] FS_Comscfd-C001014000
    [21] FS_Comscfd-C0f1018000
    [22] FS_Comscfd-C001020000
    [23] FS_Comscfd-C001021000
    [24] FS_Comscfd-C001000000
    [25] FS_Comscfd-C002003000
    [26] FS_Comscfd-C002006000
    [27] FS_Comscfd-C002007000
    [28] FS_Comscfd-C002000000
    [29] FS_Comscfd-C003003000
    [30] FS_Comscfd-C003002000
    [31] FS_Comscfd-C003005000
    [32] FS_Comscfd-C003006000
    [33] FS_Comscfd-C003000000
    [

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  列数: 32
    [ 0] code
    [ 1] stknme
    [ 2] listingDate
    [ 3] EndDate
    [ 4] FS_Combas-A001101000
    [ 5] FS_Combas-A001107000
    [ 6] FS_Combas-A001109000
    [ 7] FS_Combas-A001123000
    [ 8] FS_Combas-A001100000
    [ 9] FS_Combas-A001212000
    [10] FS_Combas-A001204000
    [11] FS_Combas-A001207000
    [12] FS_Combas-A001218000
    [13] FS_Combas-A001219000
    [14] FS_Combas-A001200000
    [15] FS_Combas-A001000000
    [16] FS_Combas-A002101000
    [17] FS_Combas-A002107000
    [18] FS_Combas-A002108000
    [19] FS_Combas-A002114000
    [20] FS_Combas-A002125000
    [21] FS_Combas-A002100000
    [22] FS_Combas-A002201000
    [23] FS_Combas-A002206000
    [24] FS_Combas-A002200000
    [25] FS_Combas-A002000000
    [26] FS_Combas-A003101000
    [27] FS_Combas-A003102000
    [28] FS_Combas-A003103000
    [29] FS_Combas-A003105000
    [30] FS_Combas-A003000000
    [31] FS_Combas-A004000000

  前 3 行数据:
   code stknme listingDate  EndDate FS_Combas-A001101000 FS_Combas-A001

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 3. 建立初步变量字典

记录每个变量所在的源文件、原始列名、清洗后变量名（暂空）、含义和单位。
后续在 Notebook 2 中完善 `clean_variable`、`definition` 和 `note` 字段。

In [14]:
dict_rows = []
for f in xlsx_files:
    df_meta = pd.read_excel(f, nrows=2, header=None)
    labels = list(df_meta.iloc[0])
    units = list(df_meta.iloc[1])
    cols = list(pd.read_excel(f, skiprows=[1,2]).columns)
    for i, col in enumerate(cols):
        label = labels[i] if i < len(labels) else ''
        unit = units[i] if i < len(units) else ''
        dict_rows.append({
            'source_file': f.name,
            'raw_variable': col,
            'clean_variable': '',
            'definition': str(label),
            'unit': str(unit),
            'note': ''
        })

var_dict = pd.DataFrame(dict_rows)
var_dict.to_csv(DICT_DIR / 'variable_dictionary.csv', index=False, encoding='utf-8-sig')
print(f'变量字典已保存: {DICT_DIR / "variable_dictionary.csv"}')
print(f'共 {len(var_dict)} 个变量')
var_dict.head(15)

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook

变量字典已保存: c:\Users\张璇\Desktop\data-analysis-group-homework\data-analysis-homework\ex_zhangxuan_02\data\dict\variable_dictionary.csv
共 217 个变量


,source_file,raw_variable,clean_variable,definition,unit,note
0,CSMAR常用变量-2000-2024.xlsx,Stkcd,,Stkcd,股票代码,
1,CSMAR常用变量-2000-2024.xlsx,accper,,accper,会计年度,
2,CSMAR常用变量-2000-2024.xlsx,stknme,,stknme,股票简称,
3,CSMAR常用变量-2000-2024.xlsx,AnaAttention,,AnaAttention,分析师关注度,
4,CSMAR常用变量-2000-2024.xlsx,Audittyp,,Audittyp,审计意见,
5,CSMAR常用变量-2000-2024.xlsx,InternationalBig4,,InternationalBig4,审计师是否来自国际四大,
6,CSMAR常用变量-2000-2024.xlsx,Ysmvosd,,Ysmvosd,年个股流通市值,
7,CSMAR常用变量-2000-2024.xlsx,Ysmvttl,,Ysmvttl,年个股总市值,
8,CSMAR常用变量-2000-2024.xlsx,Yretwd,,Yretwd,考虑现金红利再投资的年个股回报率,
9,CSMAR常用变量-2000-2024.xlsx,PropertyRightsNature,,PropertyRightsNature,产权性质,


## 4. 数据概览汇总

汇总各文件的基本信息。

In [15]:
xlsx_files = sorted(RAW_DIR.glob('*.xlsx'))
summary_rows = []
import openpyxl
for f in xlsx_files:
    wb = openpyxl.load_workbook(f, read_only=True)
    ws = wb[wb.sheetnames[0]]
    total_rows = ws.max_row - 2 if ws.max_row else 0
    cols_df = pd.read_excel(f, skiprows=2, nrows=0)
    wb.close()
    summary_rows.append({
        'filename': f.name,
        'rows': total_rows,
        'columns': len(cols_df.columns),
        'size_kb': round(f.stat().st_size / 1024, 0)
    })

summary_df = pd.DataFrame(summary_rows)
print('\n各文件数据概览：')
print(summary_df.to_string(index=False))

d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\miniconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook


各文件数据概览：
                 filename   rows  columns  size_kb
 CSMAR常用变量-2000-2024.xlsx  61457       33  10841.0
上市公司基本信息变更表2000-2024.xlsx 160277        8   9081.0
         上市公司基本信息年度表.xlsx  64172       40  26853.0
 利润表-现金流量表-2000-2010.xlsx  64165       36  10438.0
 利润表-现金流量表-2011-2024.xlsx  81664       36  20459.0
     资产负债表-2000-2010.xlsx  64165       32   9152.0
     资产负债表-2011-2024.xlsx  81664       32  17716.0


---
**Notebook 1 完成。** 原始数据已解压至 `data/raw/`，变量字典已创建。
下一步：打开 `02_clean_construct_variables.ipynb` 进行数据清洗与变量构造。